# EDA work

In [ ]:
#is it statisticall risky to note all 0 when the score_nonres is out of range of the score field? so when it is 998 or something?

In [13]:
%matplotlib inline

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import CategoricalDtype


# paths
figdir = "figs"
os.makedirs(figdir, exist_ok=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [3]:
# brings run_pipeline() into scope and runs no extra code (since you cleaned it)
%run -i data_proc.ipynb

# call with your single flag
raw_data, proc_data, admin_report = run_pipeline(drop_visit_year=False)

print("raw_data:", raw_data.shape)
print("proc_data:", proc_data.shape)
print("admin_report type:", type(admin_report))


C:\Users\miked\AppData\Local\Temp\ipykernel_28624\2710141733.py:26: DtypeWarning: Columns (20,22,24,26,28,41,44,46,48,51,61,63,65,67,69,71,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,134,156,165,176,179,189,217,220,222,224,226,228,230,232,234,236,238,240,242,244,246,248,250,252,254,256,258,260,262,264,266,268,270,272,382,397,399,401,419,421,423,432,445,454,494,574,605,613,638,674,690,704,707,710,715,727,738,744,746,804,809,810,811,812,820,831,833,835,837,843,904,959,960,961,969,970,971,972,982,1004,1007,1010) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(r"C:\Users\miked\Desktop2\RIT ISTE Data Mining\project\investigator_nacc70.csv")


removed 278 composite columns
initial shape: (204031, 746)
after keep_v3_plus: (100120, 746)
after drop_not_in_v3: (100120, 595)
after drop_text_x_columns: (100120, 561)
after clean_placeholders: (100120, 561)
after apply_score_nonres_bounds: (100120, 667)
after flag_admin_and_missing: (100120, 1219)
after fix_dtypes: (100120, 1219)
after fill_moca: (100120, 1219)
backfill_a5_from_mapping_df: filled 1161068 values across 39 columns.
 top fills: TOBAC30:35761, TOBAC100:35761, PACKSPER:35761, ALCOCCAS:35761, CVHATT:35761, CVOTHR:35761, CBSTROKE:35761, STROKMUL:35761, CBTIA:35761, TIAMULT:35761
after backfill_a5_from_mapping_df: (100120, 1219)
after combine_img_fields: (100120, 1218)
after add_visit_features: (100120, 1221)
after drop_visit_parts_after_visit_date: (100120, 1219)
after drop_sparse: (100120, 633)
after drop_leaky: (100120, 633)
Removed 250 all-zero invalid flags.
after drop_zero_invalid_flags: (100120, 383)
Final Shape: (100120, 383)
raw_data: (204031, 1024)
proc_data: (100

In [11]:
raw_data.shape

(204031, 1024)

In [ ]:
print("type:", type(administrative_report))
if isinstance(administrative_report, pd.DataFrame):
    print("shape:", administrative_report.shape)
    display(administrative_report.head(10))
else:
    print("value preview:", str(administrative_report)[:500])

In [ ]:
# show a compact view of administrative_report if it exists
try:
    display(administrative_report if hasattr(administrative_report, "head") else administrative_report)
except Exception:
    pass


In [ ]:
def quick_overview(df: pd.DataFrame, name: str):
    print(f"\n=== {name}: shape {df.shape} ===")
    print(df.info())
    miss = df.isna().mean().mul(100).sort_values(ascending=False)
    print("\nmissing % (top 40):")
    display(miss.head(40).to_frame("pct").style.format({"pct":"{:.2f}"}))

def visit_stats(df: pd.DataFrame, id_col="NACCID"):
    if id_col not in df.columns:
        print(f"visit_stats: '{id_col}' not in columns")
        return None
    cnt = df.groupby(id_col).size()
    print(f"avg visits per {id_col}: {cnt.mean():.2f}")
    print(f"std visits per {id_col}: {cnt.std():.2f}")
    return cnt

def export_missing(df: pd.DataFrame, stem: str):
    pm = df.isna().mean().mul(100)
    any_miss = pm[pm > 0.0].sort_values(ascending=False)
    any_miss_df = (
        any_miss.round(2).rename("pct_missing").to_frame()
        .reset_index().rename(columns={"index":"column"})
    )
    any_miss_df.to_csv(f"missing_gt0_{stem}.csv", index=False)

    high = pm[pm >= 15.0].sort_values(ascending=False)
    high_df = (
        high.round(2).rename("pct_missing").to_frame()
        .reset_index().rename(columns={"index":"column"})
    )
    high_df.to_csv(f"high_miss_ge15_{stem}.csv", index=False)

    bins = [-0.001, 0.0, 15.0, 50.0, 80.0, 100.0]
    labels = ["0%", "0–15%", "15–50%", "50–80%", ">=80%"]
    tiers = pd.cut(pm, bins=bins, labels=labels, include_lowest=True, right=True)
    print("columns per missingness tier:")
    display(tiers.value_counts().reindex(labels).to_frame("n"))

def class_bar(df: pd.DataFrame, y_col="NACCUDSD", title="NACCUDSD class distribution", stem="eda"):
    if y_col not in df.columns:
        print(f"class_bar: '{y_col}' not in columns")
        return
    y = pd.to_numeric(df[y_col], errors="coerce")
    order = [1, 2, 3, 4]
    label_map = {1: "Normal cognition", 2: "Impaired-not-MCI", 3: "MCI", 4: "Dementia"}
    labels = [label_map.get(k, str(k)) for k in order]
    counts = [(y == k).sum() for k in order]
    total = int(y.notna().sum())

    plt.figure(figsize=(7,4))
    plt.bar(labels, counts)
    plt.ylabel("count")
    plt.title(title)
    for i, v in enumerate(counts):
        pct = (v / total * 100) if total > 0 else 0
        plt.text(i, v, f"{v}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=9)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    out = os.path.join(figdir, f"fig_{stem}_naccudsd_bar.png")
    plt.savefig(out, dpi=150); plt.show()
    print("saved:", out)

def type_summary(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "n_rows":[len(df)],
        "n_cols":[df.shape[1]],
        "n_numeric":[df.select_dtypes(include=[np.number]).shape[1]],
        "n_category":[df.select_dtypes(include="category").shape[1]],
        "n_string":[df.select_dtypes(include="string").shape[1]],
        "n_object":[df.select_dtypes(include="object").shape[1]],
    })

# association screen (cramer's v for categoricals; mutual information for numerics)
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif

def cramers_v(x: pd.Series, y: pd.Series) -> float:
    ct = pd.crosstab(x, y)
    if ct.empty:
        return 0.0
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    if n == 0:
        return 0.0
    phi2 = chi2 / n
    r, k = ct.shape
    phi2corr = max(0.0, phi2 - ((k - 1)*(r - 1)) / (n - 1)) if n > 1 else 0.0
    rcorr = r - ((r - 1)**2) / (n - 1) if n > 1 else r
    kcorr = k - ((k - 1)**2) / (n - 1) if n > 1 else k
    denom = min((kcorr - 1), (rcorr - 1))
    return float(np.sqrt(phi2corr / denom)) if denom > 0 else 0.0

def auto_exclude_id_like(df: pd.DataFrame, base_exclude=None, thresh=0.90):
    base_exclude = base_exclude or []
    n = len(df)
    extra = []
    for c in df.columns:
        if c in base_exclude or n == 0: 
            continue
        try:
            nunq = df[c].astype(str).nunique(dropna=True)
        except Exception:
            nunq = df[c].nunique(dropna=True)
        if (nunq / n) >= thresh:
            extra.append(c)
    return sorted(set(base_exclude) | set(extra))

def assoc_with_y(df: pd.DataFrame, y_col="NACCUDSD", exclude_cols=None,
                 skip_high_card_thresh=0.50, num_min_unique=3, num_min_conv=0.50):
    exclude_cols = exclude_cols or []
    y = df[y_col].astype(str)
    cols = [c for c in df.columns if c != y_col and c not in exclude_cols]
    out = []
    n = len(df)

    for c in cols:
        s = df[c]
        s_num = pd.to_numeric(s, errors="coerce")
        conv_ratio = 1.0 - s_num.isna().mean()
        nunq = s_num.nunique(dropna=True)

        if conv_ratio >= num_min_conv and nunq >= num_min_unique:
            x = s_num.fillna(s_num.median())
            try:
                mi = mutual_info_classif(x.values.reshape(-1, 1), y, discrete_features=False, random_state=0)
                out.append((c, float(mi[0]), "mutual_info"))
                continue
            except Exception:
                pass

        s_cat = s.astype(str)
        if n > 0 and s_cat.nunique(dropna=True) / n > skip_high_card_thresh:
            continue
        try:
            v = cramers_v(s_cat, y)
        except Exception:
            v = 0.0
        out.append((c, v, "cramers_v"))

    res = pd.DataFrame(out, columns=["feature", "association", "method"]).sort_values("association", ascending=False)
    res.reset_index(drop=True, inplace=True)
    return res

def norm_assoc(df: pd.DataFrame, assoc_tbl: pd.DataFrame, y_col="NACCUDSD") -> pd.DataFrame:
    p = df[y_col].astype(str).value_counts(normalize=True)
    h_y = float(-(p * np.log(p)).sum()) if len(p) > 1 else 1.0
    s = assoc_tbl.copy()
    s["score"] = np.where(
        s["method"].astype(str).str.lower().eq("mutual_info"),
        s["association"].astype(float) / max(h_y, 1e-12),
        s["association"].astype(float)
    ).clip(0, 1)
    return s

def plot_assoc_heat(s: pd.DataFrame, top_n=15, title="association with NACCUDSD (normalized)", stem="eda"):
    t = s.sort_values("score", ascending=False).head(top_n).reset_index(drop=True)
    feats = t["feature"].tolist()
    vals = t["score"].to_numpy().reshape(-1, 1)

    fig, ax = plt.subplots(figsize=(6, 0.35 * len(feats) + 1))
    im = ax.imshow(vals, aspect="auto", vmin=0, vmax=1)
    ax.set_yticks(np.arange(len(feats))); ax.set_yticklabels(feats, fontsize=8)
    ax.set_xticks([0]); ax.set_xticklabels(["normalized association (0–1)"], fontsize=9)
    ax.set_title(title, fontsize=11, pad=10)
    cbar = fig.colorbar(im, ax=ax); cbar.set_label("score (0–1)", rotation=90, labelpad=10)
    plt.tight_layout()
    out = os.path.join(figdir, f"fig_{stem}_assoc_heat.png")
    plt.savefig(out, dpi=150); plt.show()
    print("saved:", out)

def simple_table1(df: pd.DataFrame, strata="NACCUDSD") -> pd.DataFrame:
    d = df.loc[df[strata].notna()].copy()
    s_num = pd.to_numeric(d[strata], errors="coerce")
    if s_num.notna().sum() == d[strata].notna().sum():
        levels = np.sort(s_num.dropna().unique())
        labels = [str(int(x)) for x in levels]
        masks = [(lbl, (s_num == lv)) for lbl, lv in zip(labels, levels)]
    else:
        levels = d[strata].dropna().astype(str).unique()
        labels = sorted(levels)
        masks = [(lbl, (d[strata].astype(str) == lbl)) for lbl in labels]
    col_order = labels + ["Overall"]
    rows = []

    def _mean_sd(s):
        s = pd.to_numeric(s, errors="coerce")
        return "NA" if s.notna().sum() == 0 else f"{s.mean():.2f} ({s.std(ddof=1):.2f})"
    def _median(s):
        s = pd.to_numeric(s, errors="coerce")
        return "NA" if s.notna().sum() == 0 else f"{int(round(s.median()))}"
    def _n_pct_eq(s, code):
        s = pd.to_numeric(s, errors="coerce")
        denom = int(s.notna().sum())
        if denom == 0: return "NA"
        num = int((s == code).sum()); pct = 100.0 * num / denom
        return f"{num} ({pct:.1f}%)"

    rN = {}
    for lbl, mask in masks: rN[lbl] = int(mask.sum())
    rN["Overall"] = int(d.shape[0])
    rows.append(pd.Series(rN, name="N (rows)"))

    if "MOCATOTS" in d.columns:
        r = {lbl: _mean_sd(d.loc[mask, "MOCATOTS"]) for lbl, mask in masks}; r["Overall"] = _mean_sd(d["MOCATOTS"])
        rows.append(pd.Series(r, name="MOCATOTS mean (sd)"))
    if "NACCAMD" in d.columns:
        r = {lbl: _mean_sd(d.loc[mask, "NACCAMD"]) for lbl, mask in masks}; r["Overall"] = _mean_sd(d["NACCAMD"])
        rows.append(pd.Series(r, name="NACCAMD mean (sd)"))
    if "BIRTHYR" in d.columns:
        r = {lbl: _median(d.loc[mask, "BIRTHYR"]) for lbl, mask in masks}; r["Overall"] = _median(d["BIRTHYR"])
        rows.append(pd.Series(r, name="BIRTHYR median"))
    if "RACE" in d.columns:
        r = {lbl: _n_pct_eq(d.loc[mask, "RACE"], 1) for lbl, mask in masks}; r["Overall"] = _n_pct_eq(d["RACE"], 1)
        rows.append(pd.Series(r, name="RACE = 1 n (%)"))
    if "SEX" in d.columns:
        r = {lbl: _n_pct_eq(d.loc[mask, "SEX"], 2) for lbl, mask in masks}; r["Overall"] = _n_pct_eq(d["SEX"], 2)
        rows.append(pd.Series(r, name="SEX = 2 n (%)"))
    if "EDUC" in d.columns:
        r = {lbl: _n_pct_eq(d.loc[mask, "EDUC"], 18) for lbl, mask in masks}; r["Overall"] = _n_pct_eq(d["EDUC"], 18)
        rows.append(pd.Series(r, name="EDUC = 18 n (%)"))

    t1 = pd.DataFrame(rows)[col_order]
    return t1

def add_timeline_feats(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    if "VISITMO" in d.columns and "VISITDAY" in d.columns:
        d["visit_date"] = pd.to_datetime({"year": d["VISITYR"], "month": d["VISITMO"], "day": d["VISITDAY"]}, errors="coerce")
    elif "VISITMO" in d.columns:
        d["visit_date"] = pd.to_datetime({"year": d["VISITYR"], "month": d["VISITMO"], "day": 1}, errors="coerce")
    else:
        d["visit_date"] = pd.to_datetime({"year": d["VISITYR"], "month": 6, "day": 1}, errors="coerce")
    d = d.sort_values(["NACCID", "visit_date"]).reset_index(drop=True)
    d["days_since_prev"] = d.groupby("NACCID")["visit_date"].diff().dt.days
    first_visit = d.groupby("NACCID").cumcount().eq(0)
    d.loc[first_visit, "days_since_prev"] = 0
    _nacc = pd.to_numeric(d["NACCUDSD"], errors="coerce")
    d["naccudsd_bin"] = _nacc.ne(1).where(_nacc.notna()).astype("Int64")
    return d


In [ ]:
if raw_data is None:
    print("raw_data is None (your pipeline may not return it). skipping raw EDA.")
else:
    quick_overview(raw_data, "raw_data")
    visit_stats(raw_data, "NACCID")
    if "VISITYR" in raw_data.columns:
        display(raw_data["VISITYR"].value_counts().to_frame("count").head(20))

    class_bar(raw_data, y_col="NACCUDSD", title="NACCUDSD (raw_data)", stem="raw")
    display(type_summary(raw_data))

    export_missing(raw_data, stem="raw")

    # a few simple histograms (only if present)
    plot_cols = [c for c in ["MOCATOTS","CDRSUM","NACCGDS","NACCAMD"] if c in raw_data.columns]
    for c in plot_cols:
        s = pd.to_numeric(raw_data[c], errors="coerce").dropna()
        plt.figure(figsize=(6,4))
        plt.hist(s, bins=30)
        plt.title(f"{c} distribution (raw_data)")
        plt.xlabel(c); plt.ylabel("count")
        plt.tight_layout()
        out = os.path.join(figdir, f"fig_raw_{c.lower()}_hist.png")
        plt.savefig(out, dpi=150); plt.show()
        print("saved:", out)


In [ ]:
if proc_data is None:
    raise RuntimeError("proc_data is None — make sure run_pipeline() returns it.")

quick_overview(proc_data, "proc_data")
visit_stats(proc_data, "NACCID")
for col in ["VISITYR", "PACKET", "RACE"]:
    if col in proc_data.columns:
        display(proc_data[col].value_counts(dropna=False).to_frame("count").head(20))

class_bar(proc_data, y_col="NACCUDSD", title="NACCUDSD (proc_data)", stem="proc")
display(type_summary(proc_data))

export_missing(proc_data, stem="proc")

# association screen
base_exclude = ["NACCID","NACCADC","PACKET","FORMVER","NACCAVST","NACCNVST","NACCUDSD"]
exclude = auto_exclude_id_like(proc_data, base_exclude=base_exclude, thresh=0.90)
assoc_tbl = assoc_with_y(proc_data, y_col="NACCUDSD", exclude_cols=exclude,
                         skip_high_card_thresh=0.50, num_min_unique=3, num_min_conv=0.50)
s = norm_assoc(proc_data, assoc_tbl, y_col="NACCUDSD")
display(s.head(30))
plot_assoc_heat(s, top_n=15, title="association with NACCUDSD (normalized) — proc_data", stem="proc")

# table 1
t1 = simple_table1(proc_data, strata="NACCUDSD")
display(t1)
t1.to_csv("table1_by_naccudsd_proc.csv", index=True)

# timeline features
dft = add_timeline_feats(proc_data)
peek = [c for c in ["NACCID","PACKET","visit_date","days_since_prev","NACCUDSD","naccudsd_bin"] if c in dft.columns]
display(dft[peek].head(15))
dft.to_csv("proc_with_timeline.csv", index=False)

# a few numeric distributions
plot_cols = [c for c in ["MOCATOTS","CDRSUM","NACCGDS","NACCAMD"] if c in proc_data.columns]
for c in plot_cols:
    s = pd.to_numeric(proc_data[c], errors="coerce").dropna()
    plt.figure(figsize=(6,4))
    plt.hist(s, bins=30)
    plt.title(f"{c} distribution (proc_data)")
    plt.xlabel(c); plt.ylabel("count")
    plt.tight_layout()
    out = os.path.join(figdir, f"fig_proc_{c.lower()}_hist.png")
    plt.savefig(out, dpi=150); plt.show()
    print("saved:", out)


In [7]:
proc_data['NACCUDSD'].value_counts()

NACCUDSD
1    54618
4    23468
3    17770
2     4264
Name: count, dtype: int64

In [5]:

# assume proc_data and your target column exist
y_col = "naccudsd_bin"  # 0 = non-demented, 1 = demented
id_col = "NACCID"

# select numeric columns only (to avoid object/categorical noise)
num_cols = [c for c in proc_data.columns if pd.api.types.is_numeric_dtype(proc_data[c]) and c not in [id_col]]

# compute variability (standard deviation) within each class
var_by_class = (
    proc_data.groupby(y_col)[num_cols]
    .std(numeric_only=True)
    .T  # transpose for easier viewing
    .rename(columns={0: "std_non_demented", 1: "std_demented"})
)

# add an overall std for context
var_by_class["std_overall"] = proc_data[num_cols].std()

# sort by overall variability (descending)
var_by_class_sorted = var_by_class.sort_values("std_overall", ascending=False)



Top 10 most variable numeric features:


naccudsd_bin,std_non_demented,std_demented,std_overall
NACCDAYS,1872.394057,1694.149791,1833.205656
NACCFDYS,1635.90432,1527.971407,1598.511437
INBIRYR,1159.593812,1208.709642,1182.517913
NACCYOD,517.671479,913.793515,756.921591
days_since_prev,268.916009,277.606695,273.488117
NACCSTYR,216.27792,316.619583,266.864378
HRATE,220.270759,256.348767,237.034008
BPSYS,105.705528,70.726028,92.446622
TRAILB,47.689144,98.713348,76.179904
NACCNRYR,0.0,68.630756,46.281678



Bottom 10 least variable numeric features:


naccudsd_bin,std_non_demented,std_demented,std_overall
invalid_CDRLANG,0.01419,0.009376,0.012239
invalid_COMPORT,0.01419,0.009376,0.012239
invalid_CDRSUM,0.01419,0.009376,0.012239
invalid_PERSCARE,0.01419,0.009376,0.012239
invalid_HOMEHOBB,0.01419,0.009376,0.012239
invalid_JUDGMENT,0.01419,0.009376,0.012239
invalid_ORIENT,0.01419,0.009376,0.012239
invalid_MEMORY,0.01419,0.009376,0.012239
invalid_CDRGLOB,0.01419,0.009376,0.012239
invalid_COMMUN,0.01419,0.009376,0.012239


In [11]:
# display top and bottom 10
print("Top 10 most variable numeric features:")
display(var_by_class_sorted.head(30))




Top 10 most variable numeric features:


naccudsd_bin,std_non_demented,std_demented,std_overall
NACCDAYS,1872.394057,1694.149791,1833.205656
NACCFDYS,1635.90432,1527.971407,1598.511437
INBIRYR,1159.593812,1208.709642,1182.517913
NACCYOD,517.671479,913.793515,756.921591
days_since_prev,268.916009,277.606695,273.488117
NACCSTYR,216.27792,316.619583,266.864378
HRATE,220.270759,256.348767,237.034008
BPSYS,105.705528,70.726028,92.446622
TRAILB,47.689144,98.713348,76.179904
NACCNRYR,0.0,68.630756,46.281678


In [15]:
print("\nBottom 10 least variable numeric features:")
display(var_by_class_sorted.tail(100))


Bottom 10 least variable numeric features:


naccudsd_bin,std_non_demented,std_demented,std_overall
UDSVERFN,0.986617,1.268511,1.119392
UDSVERLR,0.985306,1.209064,1.08801
IMG_TOT,0.942764,0.861808,0.910048
UDSVERNF,0.731364,0.928367,0.823198
UDSVERLN,0.598276,0.877551,0.733982
CDRGLOB,0.142838,0.811605,0.712908
TRAILARR,0.58646,0.80473,0.691775
naccudsd_bin,0.0,0.0,0.497926
invalid_MINTSCNC,0.499787,0.47819,0.496861
invalid_MOCARECR,0.491557,0.406053,0.494584


In [7]:
proc_data['NACCAGEB'].isnull().sum()

532

In [ ]:

proc_data['NACCDAYS'].isnull().sum()

In [15]:
proc_data['VISITYR'].value_counts()

VISITYR
2023    11523
2019    11001
2018    10610
2022    10388
2024    10326
2017    10139
2016     9899
2021     8869
2020     8338
2015     7945
2025     1082
Name: count, dtype: int64

In [ ]:
cat_cols = [c for c in proc_data.columns if proc_data[c].dtype == "object"]

cat_var = []
for c in cat_cols:
    freq = proc_data.groupby(y_col)[c].value_counts(normalize=True).unstack(fill_value=0)
    if freq.shape[1] > 1:
        diff = freq.diff(axis=0).iloc[-1].abs().max()  # max difference in category freq between classes
        cat_var.append((c, diff))
cat_var_df = pd.DataFrame(cat_var, columns=["variable", "max_class_diff"]).sort_values("max_class_diff", ascending=False)

print("Top 10 categorical features most different by class:")
display(cat_var_df.head(10))
